## Iterative Imputer 
often referred as **MICE** which stands for which stands for **Multivariate Imputation by Chained Equations**. Recommended in situations where you need accuracy no matter how much time it takes, as iterative imputer is better in accuracy compared to simple and knn. This comes under **advance imputation techniques**.

Instead of just looking at standard distances between rows like KNNImputer, treats every column with missing values as a target variable in a machine learning regression problem. 

**Approach:** Model-based (chained regression)

**Relationship Handling:** Captures complex linear & non-linear correlations

**Flexibility:** High (you can plug in decision trees, random forests, or linear models)

### How IterativeImputer Works Under the Hood
1) Initial Fill: It starts by filling all missing values with a placeholder (like the column mean or median).
2) Column-by-Column Regression:
    - It picks the first column with missing values and turns it into the target variable $y$.
    - It uses all other features as input predictors $X$.
    - It trains a regression model **(by default, Bayesian Ridge Regression)** on the known rows and predicts the missing values for that column.
3) Iteration (Chained Equations): It repeats this process for every column with missing values, updating the estimates step by step.
4) Convergence: It repeats this entire round-robin cycle multiple times (e.g., 10 iterations) until the predicted values stabilize and stop changing significantly.

### When to choose this technique:
- Your dataset has complex, multi-column interdependencies.
- You want maximum precision for statistical modeling or advanced predictive algorithms.
- Missing values are present across multiple features simultaneously.



In [2]:
# Example dataset

import numpy as np
import pandas as pd

# Create a sample dataset with correlated features and missing values
data = {
    'Experience_Years': [1, 2, 3, 10, 11, 12, 20, 22],
    'Age': [22, 24, 25, 33, 35, 36, 48, 52],
    'Salary': [30000, 35000, 38000, 85000, np.nan, 95000, 150000, 160000] # Missing senior-level salary
}

df = pd.DataFrame(data)

print("=== ORIGINAL DATASET WITH MISSING VALUES ===")
print(df)

=== ORIGINAL DATASET WITH MISSING VALUES ===
   Experience_Years  Age    Salary
0                 1   22   30000.0
1                 2   24   35000.0
2                 3   25   38000.0
3                10   33   85000.0
4                11   35       NaN
5                12   36   95000.0
6                20   48  150000.0
7                22   52  160000.0


In [3]:
# Enable the experimental IterativeImputer feature in scikit-learn
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Create a copy so original df stays untouched
df_iterative = df.copy()

# Initialize IterativeImputer (max_iter=10 runs 10 cycles, random_state for reproducibility)
iterative_imp = IterativeImputer(max_iter=10, random_state=42)

# Fit and transform across all features
df_iterative_filled = pd.DataFrame(
    iterative_imp.fit_transform(df_iterative),
    columns=df_iterative.columns
)

print("=== ITERATIVE IMPUTER RESULT (MICE) ===")
print(df_iterative_filled)

=== ITERATIVE IMPUTER RESULT (MICE) ===
   Experience_Years   Age         Salary
0               1.0  22.0   30000.000000
1               2.0  24.0   35000.000000
2               3.0  25.0   38000.000000
3              10.0  33.0   85000.000000
4              11.0  35.0   90599.047047
5              12.0  36.0   95000.000000
6              20.0  48.0  150000.000000
7              22.0  52.0  160000.000000


---
As by default the iterative imputer uses **BayesianRidgeRegression**, Let's see how that code looks with different models, which will show, how can use other models in this technique. so here are some examples, if you got idea about ML models:

In [5]:
# Random forest-based imputation using IterativeImputer

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Use Random Forest as the underlying regression model for imputation
rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42),
    max_iter=10,
    random_state=42
)

df_rf_imputed = pd.DataFrame(
    rf_imputer.fit_transform(df), 
    columns=df.columns
)

print("=== ITERATIVE IMPUTER (RANDOM FOREST) ===")
print(df_rf_imputed)

=== ITERATIVE IMPUTER (RANDOM FOREST) ===
   Experience_Years   Age    Salary
0               1.0  22.0   30000.0
1               2.0  24.0   35000.0
2               3.0  25.0   38000.0
3              10.0  33.0   85000.0
4              11.0  35.0   86550.0
5              12.0  36.0   95000.0
6              20.0  48.0  150000.0
7              22.0  52.0  160000.0


In [7]:
# Decision tree-based imputation using IterativeImputer

from sklearn.tree import DecisionTreeRegressor

dt_imputer = IterativeImputer(
    estimator=DecisionTreeRegressor(max_depth=5, random_state=42),
    max_iter=10,
    random_state=42
)

df_dt_imputed = pd.DataFrame(
    dt_imputer.fit_transform(df), 
    columns=df.columns
)

print("=== ITERATIVE IMPUTER (DECISION TREE) ===")
print(df_dt_imputed)

=== ITERATIVE IMPUTER (DECISION TREE) ===
   Experience_Years   Age    Salary
0               1.0  22.0   30000.0
1               2.0  24.0   35000.0
2               3.0  25.0   38000.0
3              10.0  33.0   85000.0
4              11.0  35.0   95000.0
5              12.0  36.0   95000.0
6              20.0  48.0  150000.0
7              22.0  52.0  160000.0


In [8]:
#

from sklearn.linear_model import LinearRegression

linear_imputer = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=10,
    random_state=42
)

df_linear_imputed = pd.DataFrame(
    linear_imputer.fit_transform(df), 
    columns=df.columns
)

print("=== ITERATIVE IMPUTER (LINEAR REGRESSION) ===")
print(df_linear_imputed)


=== ITERATIVE IMPUTER (LINEAR REGRESSION) ===
   Experience_Years   Age         Salary
0               1.0  22.0   30000.000000
1               2.0  24.0   35000.000000
2               3.0  25.0   38000.000000
3              10.0  33.0   85000.000000
4              11.0  35.0   90858.179375
5              12.0  36.0   95000.000000
6              20.0  48.0  150000.000000
7              22.0  52.0  160000.000000


### Let's now understand, how does this actually work with example:
The only thing you need to mainly understand is, the very first step is going to be filling the missing values with mean or median, as they just stay as place holders, and then it becomes a dataset which fits into any regressing model and predict values. Now lets see the example to get this point.

Imagine we have a dataset with three columns:

| **Row**      | **Age (X1​)** | **Experience (X2​)** | **Salary (Y)**      |
| ------------ | ------------- | -------------------- | ------------------- |
 **Person 1** | 22            | 1                    | 30,000              |
 **Person 2** | 45            | 20                   | 110,000             |
| **Person 3** | 35            | **`NaN`** (Missing)  | **`NaN`** (Missing) |

Notice that there are two columns with missing values.

#### Step 1: Preliminary Quick Fill (The Baseline)
Before training any ML models, the algorithm needs temporary placeholder numbers so it can do the math. It temporarily fills all **NaN values with a simple metric, usually the column mean:
- Average Experience of known rows: $(1 + 20) / 2 = 10.5$
- Average Salary of known rows: $(30,000 + 110,000) / 2 = 70,000$"

| **Row**      | **Age** | **Experience**    | **Salary**          |
| ------------ | ------- | ----------------- | ------------------- |
| **Person 1** | 22      | 1                 | 30,000              |
| **Person 2** | 45      | 20                | 110,000             |
| **Person 3** | 35      | **10.5** _(temp)_ | **70,000** _(temp)_ |

#### Step 2: Predict Column 1 (Experience)
Now the algorithm focuses on **imputing `Experience` properly**.

1. It isolates `Experience` as the **Target ($y$)**.
    
2. It uses `Age` and `Salary` as the **Features ($X$)**.
    
3. It trains a regression model on Person 1 and Person 2 (the known data points).
    
4. The model learns: _"Higher Age and higher Salary mean higher Experience."_
    
5. It uses this trained model to predict `Experience` for Person 3 using their `Age=35` and temporary `Salary=70,000`.
    

- **Model Output for Person 3 Experience:** Predicts **11 years** (updating from $10.5$).

Our updated dataset now:

